# Lullaby — Single Night Analysis

Deep-dive exploration of a single sleep session. Loads sensor data, assesses quality,
computes epoch features, visualizes all data streams, and runs statistical tests
comparing REM vs non-REM epochs.

**Usage:** Set `SESSION_PATH` below to a real session JSON, or leave as `None` to use mock data.

In [ ]:
import sys
sys.path.insert(0, '..')

import matplotlib
matplotlib.rcParams['figure.dpi'] = 100

from lullaby.loader import load_session, generate_mock_session
from lullaby.features import align_to_epochs
from lullaby.visualization import plot_night_overview, plot_feature_distributions, plot_epoch_heatmap
from lullaby.statistics import compare_rem_vs_non_rem, compute_correlations, stage_transition_analysis, format_statistics_table
from lullaby.quality import print_quality_report

print('Lullaby analysis toolkit loaded.')

## 1. Load Session Data

Set `SESSION_PATH` to a real exported JSON file, or leave as `None` to generate mock data.

In [ ]:
# Set to a real session file path, or None for mock data
SESSION_PATH = None  # e.g., '../data/lullaby_session_2025-03-15_23-00.json'

if SESSION_PATH:
    session = load_session(SESSION_PATH)
    print(f'Loaded real session: {session.session_id[:8]}...')
else:
    session = generate_mock_session(duration_hours=8.0, seed=42)
    print(f'Generated mock session: {session.session_id[:8]}...')

print(f'Duration: {session.duration_hours:.1f} hours')
print(f'HR samples: {len(session.heart_rate):,}')
print(f'HRV samples: {len(session.hrv):,}')
print(f'Accel samples: {len(session.accelerometer):,}')
print(f'Sonar features: {len(session.sonar):,}')
print(f'Audio features: {len(session.audio):,}')
print(f'Sleep stages: {len(session.sleep_stages)}')

## 2. Data Quality Report

In [ ]:
print_quality_report(session)

## 3. Night Overview Plot

Multi-panel view of all sensor streams with sleep stage color coding.
Red shading = REM periods.

In [ ]:
fig = plot_night_overview(session)
fig.savefig('../data/night_overview.png', dpi=150, bbox_inches='tight')
print('Saved to ../data/night_overview.png')

## 4. Epoch Alignment

Align all streams to 30-second epochs and compute derived features.

In [ ]:
epochs = align_to_epochs(session, epoch_seconds=30)
print(f'Epochs: {len(epochs)} rows x {len(epochs.columns)} columns')
print(f'\nColumns: {list(epochs.columns)}')
print(f'\nSleep stage distribution:')
print(epochs['sleep_stage'].value_counts())
epochs.head(10)

## 5. Feature Distributions by Sleep Stage

Violin plots showing how each feature separates across sleep stages.
The key question: **Do any features visually separate REM from other stages?**

In [ ]:
fig = plot_feature_distributions(epochs)
fig.savefig('../data/feature_distributions.png', dpi=150, bbox_inches='tight')
print('Saved to ../data/feature_distributions.png')

## 6. REM vs Non-REM Statistical Comparison

Mann-Whitney U tests for each feature between REM and non-REM (Light + Deep) epochs.
Bonferroni-corrected p-values and effect sizes.

In [ ]:
stats_df = compare_rem_vs_non_rem(epochs)
print(format_statistics_table(stats_df))
print(f'\nSignificant features (after Bonferroni correction):')
sig = stats_df[stats_df['significant']]
if not sig.empty:
    for _, row in sig.iterrows():
        print(f"  {row['feature']}: p={row['p_corrected']:.2e}, d={row['cohens_d']:.2f}")
else:
    print('  None — need more data or features with larger effect sizes')

## 7. Feature Correlation Matrix

Spearman correlations between features. Highly correlated features may be
redundant; uncorrelated features provide complementary information.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

corr = compute_correlations(epochs)
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, ax=ax, square=True, linewidths=0.5)
ax.set_title('Feature Correlations (Spearman)', fontsize=14)
plt.tight_layout()
fig.savefig('../data/correlations.png', dpi=150, bbox_inches='tight')
print('Saved to ../data/correlations.png')

## 8. Epoch Heatmap

Z-scored features over time, with sleep stage color bar.

In [ ]:
fig = plot_epoch_heatmap(epochs)
fig.savefig('../data/epoch_heatmap.png', dpi=150, bbox_inches='tight')
print('Saved to ../data/epoch_heatmap.png')

## 9. Sleep Stage Transition Analysis

Transition probabilities and pre-REM feature patterns.

In [ ]:
transitions = stage_transition_analysis(epochs)

print(f"REM onset events: {transitions['rem_onset_count']}")
print(f'\nTransition probabilities:')
print(transitions['transition_matrix'].to_string())

if not transitions['pre_rem_features'].empty:
    print(f'\nMean features in 2 epochs BEFORE REM onset:')
    print(transitions['pre_rem_features'].to_string())
    print(f'\nMean features in first 2 epochs OF REM:')
    print(transitions['post_rem_features'].to_string())

## 10. Key Findings

Summary of the most discriminative features for REM detection.

In [ ]:
print('=' * 50)
print('  KEY FINDINGS')
print('=' * 50)

if not stats_df.empty:
    print(f'\nTotal features tested: {len(stats_df)}')
    print(f'Significant after Bonferroni: {stats_df["significant"].sum()}')
    
    # Top 5 features by effect size
    top = stats_df.nlargest(5, 'cohens_d', keep='first')
    print(f'\nTop 5 features by effect size (|Cohen d|):')
    for _, row in top.iterrows():
        direction = 'higher' if row['rem_median'] > row['non_rem_median'] else 'lower'
        print(f"  {row['feature']}: d={row['cohens_d']:.2f} (REM {direction})")
    
    print(f'\nInterpretation guide:')
    print(f'  |d| > 0.8: Large effect — strong REM discriminator')
    print(f'  |d| 0.5-0.8: Medium effect — useful signal')
    print(f'  |d| 0.2-0.5: Small effect — weak but may help in combination')
    print(f'  |d| < 0.2: Negligible — not useful for REM detection')
else:
    print('\nNo statistical results available.')
    print('This could mean sleep stages are missing — sync from HealthKit first.')